# 🐍 Módulo: Estructuras de control
## RA5 · Excepciones y código robusto

**Resultado de aprendizaje:** *Crea arquitectura de código de forma eficiente y escribe código robusto.*

### Criterios de evaluación que se trabajan

| Criterio | Descripción | Apartado |
|---|---|---|
| a | Diferenciar entre excepción y error de sintaxis | 1 |
| b | Escribir instrucciones de captura de excepciones | 2 |
| c | Capturar y tratar excepciones | 3 |
| d | Tratar excepciones (lanzar, propagar, personalizar) | 4 |
| e | Realizar depuraciones de excepciones correctamente | 5 |
| f | Escribir bloques de código robusto con las sentencias adecuadas | 6 |

## 1. Excepciones frente a errores de sintaxis (criterio a)

Hay dos grandes momentos en los que algo puede fallar:

| | Error de **sintaxis** | **Excepción** |
|---|---|---|
| ¿Cuándo se detecta? | **Antes** de ejecutar (al compilar/analizar el código) | **Durante** la ejecución |
| Causa | El código no respeta la gramática de Python | El código es válido pero ocurre algo imprevisto (dato erróneo, fichero inexistente...) |
| ¿Se ejecuta parte del programa? | **No**, no se ejecuta nada | Sí, hasta que se produce la excepción |
| ¿Se puede capturar? | No directamente con `try` en el mismo archivo | **Sí**, con `try/except` |
| Ejemplos | `SyntaxError`, `IndentationError` | `ZeroDivisionError`, `ValueError`, `TypeError`... |

In [1]:
# Error de sintaxis: Python ni siquiera empieza a ejecutar este código.
# Lo comprobamos con compile(), que solo analiza el código sin ejecutarlo.
codigo_malo = "print('hola'"           # falta cerrar el paréntesis

try:
    compile(codigo_malo, "<ejemplo>", "exec")
except SyntaxError as e:
    print("Error de sintaxis detectado ANTES de ejecutar:")
    print("  ", type(e).__name__, "->", e.msg)

Error de sintaxis detectado ANTES de ejecutar:
   SyntaxError -> '(' was never closed


In [2]:
# Excepción: el código es sintácticamente correcto, pero falla al ejecutarse
print("Esta línea SÍ se ejecuta")
try:
    resultado = 10 / 0
except ZeroDivisionError as e:
    print("Excepción en tiempo de ejecución:", e)

Esta línea SÍ se ejecuta
Excepción en tiempo de ejecución: division by zero


In [3]:
# Un código con sintaxis válida pero con errores lógicos o de ejecución
casos = {
    "ZeroDivisionError": lambda: 1 / 0,
    "ValueError": lambda: int("abc"),
    "TypeError": lambda: "2" + 2,
    "IndexError": lambda: [1, 2, 3][10],
    "KeyError": lambda: {"a": 1}["b"],
    "NameError": lambda: variable_inexistente,
    "AttributeError": lambda: "texto".no_existe(),
    "FileNotFoundError": lambda: open("no_existe.txt"),
    "ModuleNotFoundError": lambda: __import__("modulo_que_no_existe"),
}

for nombre, accion in casos.items():
    try:
        accion()
    except Exception as e:
        print(f"{type(e).__name__:20} -> {e}")

ZeroDivisionError    -> division by zero
ValueError           -> invalid literal for int() with base 10: 'abc'
TypeError            -> can only concatenate str (not "int") to str
IndexError           -> list index out of range
KeyError             -> 'b'
NameError            -> name 'variable_inexistente' is not defined
AttributeError       -> 'str' object has no attribute 'no_existe'
FileNotFoundError    -> [Errno 2] No such file or directory: 'no_existe.txt'
ModuleNotFoundError  -> No module named 'modulo_que_no_existe'


### 1.1 Jerarquía de excepciones (resumen)

```
BaseException
 ├── KeyboardInterrupt
 ├── SystemExit
 └── Exception                  <- de aquí cuelgan casi todas
      ├── ArithmeticError  ── ZeroDivisionError, OverflowError
      ├── LookupError      ── IndexError, KeyError
      ├── OSError          ── FileNotFoundError, PermissionError
      ├── ValueError
      ├── TypeError
      ├── NameError
      ├── AttributeError
      └── RuntimeError     ── RecursionError
```

Capturar una clase captura también todas sus **subclases**.

In [4]:
print(ZeroDivisionError.__mro__)
print(issubclass(KeyError, LookupError), issubclass(FileNotFoundError, OSError))

(<class 'ZeroDivisionError'>, <class 'ArithmeticError'>, <class 'Exception'>, <class 'BaseException'>, <class 'object'>)
True True


### 1.2 Leer un *traceback*

Cuando una excepción no se captura, Python muestra la **traza** (de la llamada más antigua a la más reciente). Lo más importante: **la última línea** (tipo y mensaje) y la línea de tu código donde ocurrió.

In [1]:
import traceback

def a():
    return b()

def b():
    return 1 / 0

try:
    a()
except ZeroDivisionError:
    traceback.print_exc()     # muestra la traza sin detener el cuaderno

Traceback (most recent call last):
  File "C:\Users\dpinjim2703\AppData\Local\Temp\ipykernel_7652\3959254484.py", line 10, in <module>
    a()
    ~^^
  File "C:\Users\dpinjim2703\AppData\Local\Temp\ipykernel_7652\3959254484.py", line 4, in a
    return b()
  File "C:\Users\dpinjim2703\AppData\Local\Temp\ipykernel_7652\3959254484.py", line 7, in b
    return 1 / 0
           ~~^~~
ZeroDivisionError: division by zero


### ✏️ Actividad 1
Clasifica cada caso como **error de sintaxis** o **excepción** y, si es excepción, indica su tipo:

1. `print("Hola)`
2. `int("3.5")`
3. `lista = [1, 2]; lista[5]`
4. `if x = 3:`
5. `3 + "3"`
6. `for i in range(3)` (sin dos puntos)
7. `d = {}; d["x"]`

In [2]:
respuestas = {
    1: "SyntaxError",  # TU RESPUESTA
    2: "ValueError",
    3: "IndexError",
    4: "SyntaxError",
    5: "TypeError",
    6: "SyntaxError",
    7: "KeyError",
}
print(respuestas)

{1: 'SyntaxError', 2: 'ValueError', 3: 'IndexError', 4: 'SyntaxError', 5: 'TypeError', 6: 'SyntaxError', 7: 'KeyError'}


## 2. Instrucciones de captura de excepciones (criterio b)

```python
try:
    # código que puede fallar
except TipoDeExcepcion:
    # qué hacer si falla
else:
    # se ejecuta si NO hubo excepción
finally:
    # se ejecuta SIEMPRE
```

```mermaid
flowchart TD
    A[try] --> B{¿Excepción?}
    B -- No --> C[else]
    B -- Sí --> D[except que coincida]
    C --> E[finally]
    D --> E
    E --> F[Continúa el programa]
```

In [3]:
# try / except básico
try:
    numero = int("42")
    print("Convertido:", numero)
except ValueError:
    print("Eso no es un número")

try:
    numero = int("cuarenta y dos")
    print("Convertido:", numero)
except ValueError:
    print("Eso no es un número")

print("El programa continúa")

Convertido: 42
Eso no es un número
El programa continúa


In [4]:
# Sin captura, el programa se detendría. Con captura, se recupera.
def dividir(a, b):
    try:
        return a / b
    except ZeroDivisionError:
        print("No se puede dividir entre cero")
        return None

print(dividir(10, 2))
print(dividir(10, 0))

5.0
No se puede dividir entre cero
None


In [8]:
# else y finally
def leer_numero(texto):
    try:
        n = int(texto)
    except ValueError:
        print(f"  '{texto}' no es válido")
        return None
    else:
        print(f"  '{texto}' convertido correctamente")   # solo si no hubo excepción
        return n
    finally:
        print("  (fin del intento)")                     # siempre

leer_numero("7")
leer_numero("x")

  '7' convertido correctamente
  (fin del intento)
  'x' no es válido
  (fin del intento)


### 2.1 Varias excepciones

In [5]:
def operar(lista, indice, divisor):
    try:
        valor = lista[indice]
        return valor / divisor
    except IndexError:
        return "Índice fuera de rango"
    except ZeroDivisionError:
        return "División por cero"
    except TypeError:
        return "Tipos incompatibles"

print(operar([10, 20], 1, 5))
print(operar([10, 20], 5, 5))
print(operar([10, 20], 1, 0))
print(operar([10, "a"], 1, 2))

4.0
Índice fuera de rango
División por cero
Tipos incompatibles


In [11]:
# Agrupar varias excepciones en una sola cláusula
for dato in ["12", "abc", None, 3.7]:
    try:
        print(int(dato), end=" | ")
    except (ValueError, TypeError) as e:
        print(f"{type(e).__name__}", end=" | ")
print()

12 | ValueError | TypeError | 3 | 


> ⚠️ El **orden** importa: se ejecuta el **primer** `except` que coincida. Pon primero las excepciones **específicas** y al final las generales.

In [12]:
try:
    {}["x"]
except LookupError:
    print("Capturada como LookupError (clase padre)")
except KeyError:
    print("Esto NUNCA se ejecutará (KeyError ya está cubierta por LookupError)")

Capturada como LookupError (clase padre)


### ✏️ Actividad 2
Escribe una función `convertir_a_entero(valor)` que devuelva el entero correspondiente o `None` si no es posible. Debe funcionar con `"12"`, `"abc"`, `None`, `7.9`, `"3.5"`.

In [14]:
def convertir_a_entero(valor):
    try:
        n = int(valor)
    except (ValueError, TypeError):
        return None
    else:
        return n

for v in ["12", "abc", None, 7.9, "3.5"]:
    print(repr(v), "->", convertir_a_entero(v))

'12' -> 12
'abc' -> None
None -> None
7.9 -> 7
'3.5' -> None


## 3. Capturar y tratar excepciones (criterio c)

Capturar no basta: hay que **tratar** la excepción con una estrategia clara.

| Estrategia | Ejemplo |
|---|---|
| **Recuperarse** con un valor por defecto | Usar 0 si el dato no es válido |
| **Reintentar** | Volver a pedir un dato |
| **Informar** al usuario/registrar y continuar | Mensaje claro, log |
| **Limpiar recursos** y propagar | `finally`, `with` |
| **Abortar** de forma controlada | Mensaje y salida |

### 3.1 Obtener información de la excepción

In [15]:
try:
    int("hola")
except ValueError as e:
    print("Tipo   :", type(e).__name__)
    print("Mensaje:", e)
    print("Args   :", e.args)

Tipo   : ValueError
Mensaje: invalid literal for int() with base 10: 'hola'
Args   : ("invalid literal for int() with base 10: 'hola'",)


In [6]:
# Estrategia 1: valor por defecto
def a_float(texto, por_defecto=0.0):
    try:
        return float(texto)
    except ValueError:
        return por_defecto

print(a_float("3.14"), a_float("n/a"), a_float("n/a", -1))

3.14 0.0 -1


In [17]:
# Estrategia 2: reintentar hasta obtener un dato válido (entradas simuladas)
intentos = iter(["abc", "-5", "x", "25"])

def pedir_edad():
    while True:
        texto = next(intentos)                # en un programa real: input("Edad: ")
        try:
            edad = int(texto)
            if edad < 0:
                raise ValueError("la edad no puede ser negativa")
        except ValueError as e:
            print(f"  Entrada '{texto}' no válida ({e}). Inténtalo de nuevo.")
        else:
            return edad

print("Edad:", pedir_edad())

  Entrada 'abc' no válida (invalid literal for int() with base 10: 'abc'). Inténtalo de nuevo.
  Entrada '-5' no válida (la edad no puede ser negativa). Inténtalo de nuevo.
  Entrada 'x' no válida (invalid literal for int() with base 10: 'x'). Inténtalo de nuevo.
Edad: 25


In [18]:
# Estrategia 3: tratar el error y seguir con el resto de datos
registros = ["10", "20", "x", "40", "", "50"]
validos = []
errores = []

for r in registros:
    try:
        validos.append(int(r))
    except ValueError:
        errores.append(r)

print("Válidos:", validos)
print("Errores:", errores)
print("Suma de los válidos:", sum(validos))

Válidos: [10, 20, 40, 50]
Errores: ['x', '']
Suma de los válidos: 120


In [19]:
# Estrategia 4: ficheros y limpieza de recursos
def leer_primera_linea(ruta):
    try:
        with open(ruta, encoding="utf-8") as f:      # 'with' cierra el fichero siempre
            return f.readline().strip()
    except FileNotFoundError:
        return f"El fichero '{ruta}' no existe"
    except PermissionError:
        return f"Sin permisos para leer '{ruta}'"

print(leer_primera_linea("no_existe.txt"))

# Creamos un fichero de prueba y lo leemos
with open("prueba.txt", "w", encoding="utf-8") as f:
    f.write("Primera línea de prueba\nSegunda\n")
print(leer_primera_linea("prueba.txt"))

El fichero 'no_existe.txt' no existe
Primera línea de prueba


### 3.2 Lo que NO hay que hacer

In [20]:
# ❌ Mal: 'except' desnudo que oculta cualquier error (incluso Ctrl+C)
try:
    resultado = 10 / 0
except:
    pass          # el error desaparece sin dejar rastro

# ❌ Mal: capturar Exception y no hacer nada útil
# ✅ Bien: capturar lo más específico posible y tratar / registrar el error
try:
    resultado = 10 / 0
except ZeroDivisionError as e:
    print("Aviso: división por cero; se usa 0 como resultado")
    resultado = 0

Aviso: división por cero; se usa 0 como resultado


### ✏️ Actividad 3
Dada la lista de cadenas `datos = ["3", "x", "8", "", "5.5", "10"]`:

1. Convierte cada elemento a `float` usando `try/except` y calcula la **media** de los válidos.
2. Cuenta cuántos fallaron y de qué **tipo** de excepción.
3. Si no hay ningún valor válido, la media debe ser `None` (sin lanzar excepción).

In [33]:
datos = ["3", "x", "8", "", "5.5", "10"]

validos = []
fallos_por_tipo = {}
total_fallos = 0

for d in datos:
    try:
        validos.append(float(d))
    except (ValueError, TypeError) as e:
        total_fallos += 1
        tipo = type(e).__name__
        fallos_por_tipo[tipo] = fallos_por_tipo.get(tipo, 0) + 1

media = sum(validos) / len(validos) if validos else None

print("Válidos:", validos)
print("Media:", media)
print("Total fallos:", total_fallos)
print("Fallos por tipo:", fallos_por_tipo)

Válidos: [3.0, 8.0, 5.5, 10.0]
Media: 6.625
Total fallos: 2
Fallos por tipo: {'ValueError': 2}


## 4. Tratamiento avanzado: lanzar, relanzar y crear excepciones (criterio d)

### 4.1 Lanzar excepciones con `raise`

Tú también puedes **provocar** una excepción cuando detectas un problema (validación).

In [34]:
def calcular_raiz(n):
    if not isinstance(n, (int, float)):
        raise TypeError("n debe ser numérico")
    if n < 0:
        raise ValueError("n no puede ser negativo")
    return n ** 0.5

for valor in (16, -4, "a"):
    try:
        print(calcular_raiz(valor))
    except (ValueError, TypeError) as e:
        print(f"{type(e).__name__}: {e}")

4.0
ValueError: n no puede ser negativo
TypeError: n debe ser numérico


In [35]:
# Propagación: si nadie la captura, la excepción 'sube' por la pila de llamadas
def nivel3():
    raise RuntimeError("fallo en nivel 3")

def nivel2():
    nivel3()

def nivel1():
    try:
        nivel2()
    except RuntimeError as e:
        print("Capturada en nivel1:", e)

nivel1()

Capturada en nivel1: fallo en nivel 3


In [37]:
# Relanzar (raise sin argumentos) tras hacer algo, y encadenar (raise ... from ...)
def procesar(dato):
    try:
        return int(dato)
    except ValueError as e:
        print(f"  (registro) dato problemático: {dato!r}")
        raise                      # vuelve a lanzar la misma excepción

try:
    procesar("x")
except ValueError:
    print("Capturada fuera tras relanzar")

def cargar_edad(texto):
    try:
        return int(texto)
    except ValueError as e:
        raise ValueError(f"Edad no válida: {texto!r}") from e

try:
    cargar_edad("veinte")
except ValueError as e:
    print(e, "| causa original:", repr(e.__cause__))

  (registro) dato problemático: 'x'
Capturada fuera tras relanzar
Edad no válida: 'veinte' | causa original: ValueError("invalid literal for int() with base 10: 'veinte'")


### 4.2 Excepciones personalizadas

Crear tus propias excepciones hace el código más **expresivo** y permite capturar errores de tu dominio.

In [38]:
class SaldoInsuficienteError(Exception):
    """Se lanza cuando se intenta retirar más dinero del disponible."""
    def __init__(self, saldo, importe):
        self.saldo = saldo
        self.importe = importe
        super().__init__(f"Saldo {saldo} € insuficiente para retirar {importe} €")

class CuentaBloqueadaError(Exception):
    pass

class Cuenta:
    def __init__(self, saldo, bloqueada=False):
        self.saldo = saldo
        self.bloqueada = bloqueada

    def retirar(self, importe):
        if self.bloqueada:
            raise CuentaBloqueadaError("La cuenta está bloqueada")
        if importe <= 0:
            raise ValueError("El importe debe ser positivo")
        if importe > self.saldo:
            raise SaldoInsuficienteError(self.saldo, importe)
        self.saldo -= importe
        return self.saldo

cuenta = Cuenta(100)
for importe in (30, 100, -5):
    try:
        print("Nuevo saldo:", cuenta.retirar(importe))
    except SaldoInsuficienteError as e:
        print("Error:", e, "| faltan", e.importe - e.saldo, "€")
    except ValueError as e:
        print("Error de valor:", e)

try:
    Cuenta(50, bloqueada=True).retirar(10)
except CuentaBloqueadaError as e:
    print("Error:", e)

Nuevo saldo: 70
Error: Saldo 70 € insuficiente para retirar 100 € | faltan 30 €
Error de valor: El importe debe ser positivo
Error: La cuenta está bloqueada


### 4.3 `finally` y recursos

`finally` garantiza la limpieza incluso si hay `return` o una excepción no capturada.

In [39]:
def con_finally():
    try:
        print("  trabajando...")
        return "resultado"
    finally:
        print("  limpieza (siempre se ejecuta)")

print(con_finally())

  trabajando...
  limpieza (siempre se ejecuta)
resultado


In [40]:
# Gestor de contexto (with): la forma elegante de garantizar la limpieza
class Recurso:
    def __enter__(self):
        print("  recurso abierto")
        return self
    def __exit__(self, tipo, valor, traza):
        print("  recurso cerrado")
        return False         # no suprime la excepción

try:
    with Recurso():
        print("  usando recurso")
        raise ValueError("problema mientras se usaba")
except ValueError as e:
    print("Capturada:", e)

  recurso abierto
  usando recurso
  recurso cerrado
Capturada: problema mientras se usaba


### ✏️ Actividad 4
1. Crea una función `validar_edad(edad)` que lance `TypeError` si no es entero y `ValueError` si no está entre 0 y 120. Pruébala con varios valores.
2. Crea una excepción personalizada `StockInsuficienteError` y una función `vender(stock, cantidad)` que la lance si no hay unidades suficientes.

In [41]:
# TU CÓDIGO AQU# 1. Función validar_edad(edad)
def validar_edad(edad):
    if isinstance(edad, bool) or not isinstance(edad, int):
        raise TypeError(f"La edad debe ser un entero, no {type(edad).__name__}")
    if edad < 0 or edad > 120:
        raise ValueError(f"La edad {edad} debe estar entre 0 y 120")
    return edad

for prueba in (21, 0, 120, -1, 150, "21", 18.5, None, True):
    try:
        print("Edad válida:", validar_edad(prueba))
    except (TypeError, ValueError) as e:
        print(f"Error con {prueba!r} ({type(e).__name__}): {e}")


# 2. Excepción personalizada StockInsuficienteError y función vender(stock, cantidad)
class StockInsuficienteError(Exception):
    """Se lanza cuando se intenta vender más unidades de las disponibles en stock."""
    def __init__(self, stock, cantidad):
        self.stock = stock
        self.cantidad = cantidad
        super().__init__(f"Stock insuficiente ({stock} disponibles) para vender {cantidad} unidades")

def vender(stock, cantidad):
    if cantidad > stock:
        raise StockInsuficienteError(stock, cantidad)
    return stock - cantidad

for stock_actual, uds in ((10, 4), (3, 5), (5, 5)):
    try:
        restante = vender(stock_actual, uds)
        print(f"Venta realizada ({uds} uds). Stock restante: {restante}")
    except StockInsuficienteError as e:
        print(f"Error de stock: {e} | Faltan {e.cantidad - e.stock} unidades")

Edad válida: 21
Edad válida: 0
Edad válida: 120
Error con -1 (ValueError): La edad -1 debe estar entre 0 y 120
Error con 150 (ValueError): La edad 150 debe estar entre 0 y 120
Error con '21' (TypeError): La edad debe ser un entero, no str
Error con 18.5 (TypeError): La edad debe ser un entero, no float
Error con None (TypeError): La edad debe ser un entero, no NoneType
Error con True (TypeError): La edad debe ser un entero, no bool
Venta realizada (4 uds). Stock restante: 6
Error de stock: Stock insuficiente (3 disponibles) para vender 5 unidades | Faltan 2 unidades
Venta realizada (5 uds). Stock restante: 0


## 5. Depuración de excepciones (criterio e)

**Depurar** (*debugging*) es localizar y corregir la causa de un error. Método:

1. **Reproducir** el error con un caso mínimo.
2. **Leer el traceback** (última línea = qué; líneas anteriores = dónde).
3. **Formular hipótesis** y comprobarlas.
4. **Corregir** y **volver a probar** (idealmente con un test).

### 5.1 Herramientas

| Herramienta | Uso |
|---|---|
| `print()` / f-strings con `!r` | Ver el estado de las variables |
| `traceback` | Obtener/mostrar la traza de una excepción capturada |
| `logging` | Registro profesional con niveles (DEBUG, INFO, WARNING, ERROR) |
| `assert` | Comprobar condiciones que **deben** cumplirse |
| `breakpoint()` / `pdb` | Depurador interactivo línea a línea |
| Depurador del IDE (VS Code, PyCharm) | Puntos de ruptura, inspección de variables |

In [42]:
# Depuración con prints: localizar un fallo
def media_notas(notas):
    total = 0
    for nota in notas:
        print(f"DEBUG nota={nota!r} tipo={type(nota).__name__}")   # ← pista
        total += nota
    return total / len(notas)

try:
    media_notas([7, 8, "9"])
except TypeError as e:
    print("TypeError:", e)
    print("→ Hipótesis: hay una cadena mezclada con números.")

DEBUG nota=7 tipo=int
DEBUG nota=8 tipo=int
DEBUG nota='9' tipo=str
TypeError: unsupported operand type(s) for +=: 'int' and 'str'
→ Hipótesis: hay una cadena mezclada con números.


In [43]:
# Corrección: convertir los datos y tratar los no válidos
def media_notas_robusta(notas):
    validas = []
    for nota in notas:
        try:
            validas.append(float(nota))
        except (ValueError, TypeError):
            print(f"  Nota ignorada: {nota!r}")
    if not validas:
        raise ValueError("No hay notas válidas")
    return sum(validas) / len(validas)

print(media_notas_robusta([7, 8, "9", "x", None]))

  Nota ignorada: 'x'
  Nota ignorada: None
8.0


In [44]:
# traceback: obtener información detallada de una excepción capturada
import traceback, sys

def falla():
    lista = [1, 2, 3]
    return lista[10]

try:
    falla()
except IndexError as e:
    tipo, valor, tb = sys.exc_info()
    print("Tipo   :", tipo.__name__)
    print("Mensaje:", valor)
    ultimo = traceback.extract_tb(tb)[-1]
    print(f"Línea {ultimo.lineno} en '{ultimo.name}': {ultimo.line}")

Tipo   : IndexError
Mensaje: list index out of range
Línea 6 en 'falla': return lista[10]


In [45]:
# logging: registro en lugar de prints
import logging, io

flujo = io.StringIO()                 # para mostrar aquí el log (normalmente va a un fichero o consola)
logging.basicConfig(stream=flujo, level=logging.DEBUG,
                    format="%(levelname)s - %(funcName)s - %(message)s", force=True)

def dividir_log(a, b):
    logging.debug("Dividiendo %s entre %s", a, b)
    try:
        return a / b
    except ZeroDivisionError:
        logging.exception("Error al dividir")        # incluye la traza
        return None

dividir_log(10, 2)
dividir_log(1, 0)
print(flujo.getvalue())

DEBUG - dividir_log - Dividiendo 10 entre 2
DEBUG - dividir_log - Dividiendo 1 entre 0
ERROR - dividir_log - Error al dividir
Traceback (most recent call last):
  File "C:\Users\dpinjim2703\AppData\Local\Temp\ipykernel_5012\945686292.py", line 11, in dividir_log
    return a / b
           ~~^~~
ZeroDivisionError: division by zero



In [46]:
# assert: comprobar invariantes durante el desarrollo
def aplicar_descuento(precio, porcentaje):
    assert precio >= 0, f"precio negativo: {precio}"
    assert 0 <= porcentaje <= 100, f"porcentaje fuera de rango: {porcentaje}"
    return precio * (1 - porcentaje / 100)

try:
    aplicar_descuento(100, 150)
except AssertionError as e:
    print("AssertionError:", e)

print(aplicar_descuento(100, 20))

AssertionError: porcentaje fuera de rango: 150
80.0


### 5.2 Depurador interactivo (`pdb`)

Pon `breakpoint()` donde quieras parar la ejecución. En el prompt `(Pdb)` puedes usar:

| Comando | Acción |
|---|---|
| `n` | Siguiente línea (*next*) |
| `s` | Entrar en una función (*step*) |
| `c` | Continuar hasta el siguiente punto de ruptura |
| `p variable` | Mostrar una variable |
| `l` | Mostrar el código alrededor |
| `q` | Salir |

```python
def ejemplo(x):
    y = x * 2
    breakpoint()      # <- se detiene aquí (no lo ejecutamos en este cuaderno)
    return y / 0
```

En un cuaderno Jupyter también puedes usar la *magia* `%debug` justo después de una excepción para depurar post-mortem.

### 5.3 Ejercicio de depuración

El código siguiente contiene **varios errores**. Encuéntralos usando lo aprendido (lee el mensaje, añade `print`, corrige).

In [47]:
def resumen_ventas(ventas):
    """ventas: lista de diccionarios {'producto': str, 'unidades': int, 'precio': float}"""
    total = 0
    for v in ventas:
        total += v["unidades"] * v["precio"]
    return {"total": total, "media": total / len(ventas)}

ventas = [
    {"producto": "A", "unidades": 3, "precio": 9.5},
    {"producto": "B", "unidades": "2", "precio": 4.0},     # ← error de tipo
    {"producto": "C", "precio": 7.0},                      # ← falta clave
]

try:
    print(resumen_ventas(ventas))
except Exception as e:
    print(f"{type(e).__name__}: {e}")

# TU CÓDIGO AQUÍ: escribe resumen_ventas_robusto(ventas) que ignore y cuente los registros erróneos
# y devuelva también la lista de errores encontrados. Maneja el caso de lista vacía.



def resumen_ventas_robusto(ventas):
    total = 0.0
    validos = 0
    n_errores = 0
    errores = []

    for indice, v in enumerate(ventas):
        try:
            unidades = v["unidades"]
            precio = v["precio"]

            if isinstance(unidades, bool) or not isinstance(unidades, int):
                raise TypeError(f"'unidades' debe ser int, no {type(unidades).__name__}")
            if isinstance(precio, bool) or not isinstance(precio, (int, float)):
                raise TypeError(f"'precio' debe ser numérico, no {type(precio).__name__}")

            total += unidades * precio
            validos += 1
        except (KeyError, TypeError, ValueError) as e:
            n_errores += 1
            errores.append((indice, v, f"{type(e).__name__}: {e}"))

    media = (total / validos) if validos > 0 else None

    return {
        "total": total,
        "media": media,
        "validos": validos,
        "n_errores": n_errores,
        "errores": errores,
    }


print(resumen_ventas_robusto(ventas))
print(resumen_ventas_robusto([]))

TypeError: can't multiply sequence by non-int of type 'float'
{'total': 28.5, 'media': 28.5, 'validos': 1, 'n_errores': 2, 'errores': [(1, {'producto': 'B', 'unidades': '2', 'precio': 4.0}, "TypeError: 'unidades' debe ser int, no str"), (2, {'producto': 'C', 'precio': 7.0}, "KeyError: 'unidades'")]}
{'total': 0.0, 'media': None, 'validos': 0, 'n_errores': 0, 'errores': []}


## 6. Código robusto (criterio f)

**Código robusto**: sigue funcionando (o falla de forma controlada) ante entradas inesperadas, y nunca deja recursos sin liberar ni datos corruptos.

### 6.1 Principios

1. **Validar** las entradas (tipo, rango, formato).
2. **Capturar solo lo necesario** y lo más específico posible.
3. **Mensajes de error claros** que indiquen qué pasó y cómo solucionarlo.
4. **Liberar recursos** (`with`, `finally`).
5. **Fallar pronto** (cláusulas de guarda) y no ocultar errores.
6. **Registrar** (`logging`) lo ocurrido.
7. **Probar** casos normales, límite y erróneos.

### 6.2 LBYL frente a EAFP

| Estilo | Significado | Ejemplo |
|---|---|---|
| **LBYL** | *Look Before You Leap*: comprobar antes | `if clave in dic:` |
| **EAFP** | *Easier to Ask Forgiveness than Permission*: intentar y capturar (más pythónico) | `try: dic[clave] except KeyError:` |

In [48]:
dic = {"a": 1}

# LBYL
if "b" in dic:
    valor = dic["b"]
else:
    valor = 0

# EAFP
try:
    valor = dic["b"]
except KeyError:
    valor = 0

# Para diccionarios hay una opción aún más simple
valor = dic.get("b", 0)
print(valor)

0


In [49]:
# Función robusta: validación + excepciones específicas + mensajes claros
def dividir_seguro(a, b):
    """Devuelve a/b. Lanza TypeError/ValueError con mensajes claros."""
    for nombre, valor in (("a", a), ("b", b)):
        if isinstance(valor, bool) or not isinstance(valor, (int, float)):
            raise TypeError(f"'{nombre}' debe ser numérico, no {type(valor).__name__}")
    if b == 0:
        raise ValueError("El divisor no puede ser cero")
    return a / b

for args in [(10, 4), (1, 0), ("1", 2), (5, None)]:
    try:
        print(args, "->", dividir_seguro(*args))
    except (TypeError, ValueError) as e:
        print(args, "-> Error:", e)

(10, 4) -> 2.5
(1, 0) -> Error: El divisor no puede ser cero
('1', 2) -> Error: 'a' debe ser numérico, no str
(5, None) -> Error: 'b' debe ser numérico, no NoneType


In [50]:
# Programa robusto completo: lectura de un fichero CSV sencillo con datos erróneos
contenido = """ana,8.5
luis,abc
marta,9
,7
pedro,4.5,extra
"""
with open("notas.csv", "w", encoding="utf-8") as f:
    f.write(contenido)

def cargar_notas(ruta):
    notas, errores = {}, []
    try:
        with open(ruta, encoding="utf-8") as f:
            for n_linea, linea in enumerate(f, start=1):
                try:
                    nombre, nota = linea.strip().split(",")
                    if not nombre:
                        raise ValueError("nombre vacío")
                    notas[nombre] = float(nota)
                except ValueError as e:
                    errores.append((n_linea, linea.strip(), str(e)))
    except FileNotFoundError:
        raise FileNotFoundError(f"No se encontró '{ruta}'") from None
    return notas, errores

notas, errores = cargar_notas("notas.csv")
print("Notas válidas:", notas)
print("Errores:")
for e in errores:
    print("  línea", e[0], "->", e[1], "|", e[2])

Notas válidas: {'ana': 8.5, 'marta': 9.0}
Errores:
  línea 2 -> luis,abc | could not convert string to float: 'abc'
  línea 4 -> ,7 | nombre vacío
  línea 5 -> pedro,4.5,extra | too many values to unpack (expected 2)


In [51]:
# Robustez ante recursos: reintentos con espera creciente (simulado)
import time

def operacion_inestable(intento):
    if intento < 3:
        raise ConnectionError(f"fallo de red en el intento {intento}")
    return "datos recibidos"

def con_reintentos(funcion, maximo=5):
    for intento in range(1, maximo + 1):
        try:
            return funcion(intento)
        except ConnectionError as e:
            print(f"  {e}")
            if intento == maximo:
                raise                      # agotados los reintentos: propaga
            time.sleep(0.05 * intento)     # espera creciente

print(con_reintentos(operacion_inestable))

  fallo de red en el intento 1
  fallo de red en el intento 2
datos recibidos


### ✏️ Actividades finales

**A.** Escribe `leer_enteros(ruta)` que lea un fichero con un número por línea, ignore las líneas inválidas y devuelva la lista de enteros. Debe manejar `FileNotFoundError`.

**B.** Escribe una **calculadora robusta** `calcular(expresion)` que reciba textos como `"8 / 2"` o `"3 * x"` y devuelva el resultado o un mensaje de error claro para: operador no soportado, operandos no numéricos, división por cero, formato incorrecto.

**C.** Crea una clase `Inventario` con una excepción propia `ProductoNoEncontradoError` y métodos `añadir`, `retirar` y `consultar` que validen sus entradas.

In [53]:
# A
def leer_enteros(ruta):
    """Lee un fichero con un entero por línea, ignora líneas inválidas y maneja FileNotFoundError."""
    enteros = []
    try:
        with open(ruta, encoding="utf-8") as f:
            for linea in f:
                texto = linea.strip()
                if not texto:
                    continue
                try:
                    enteros.append(int(texto))
                except ValueError:
                    # Se ignora la línea si no representa un entero válido
                    pass
    except FileNotFoundError as e:
        print(f"Aviso: no se encontró el fichero '{ruta}' ({e})")
        return []
    return enteros


# Pruebas de ejemplo
with open("numeros_prueba.txt", "w", encoding="utf-8") as f:
    f.write("10\n\n-4\nabc\n3.5\n25\n")

print("Fichero válido:", leer_enteros("numeros_prueba.txt"))
print("Fichero inexistente:", leer_enteros("no_existe_enteros.txt"))

Fichero válido: [10, -4, 25]
Aviso: no se encontró el fichero 'no_existe_enteros.txt' ([Errno 2] No such file or directory: 'no_existe_enteros.txt')
Fichero inexistente: []


In [54]:
# B
def calcular(expresion):
    """Evalúa expresiones sencillas 'a op b' y devuelve resultado o mensaje de error claro."""
    if not isinstance(expresion, str):
        return "Error de formato: la expresión debe ser un texto (ejemplo: '8 / 2')."

    partes = expresion.strip().split()
    if len(partes) != 3:
        return "Error de formato: usa el formato '<numero> <operador> <numero>' (ejemplo: '8 / 2')."

    texto_a, operador, texto_b = partes

    try:
        a = float(texto_a)
        b = float(texto_b)
    except ValueError:
        return f"Error de operandos: '{texto_a}' y '{texto_b}' deben ser números válidos."

    if operador == "+":
        return a + b
    elif operador == "-":
        return a - b
    elif operador == "*":
        return a * b
    elif operador == "/":
        if b == 0:
            return "Error matemático: no se puede dividir entre cero."
        return a / b
    else:
        return f"Error de operador: '{operador}' no está soportado. Usa +, -, * o /."


# Pruebas de ejemplo
for expr in ["8 / 2", "3 * x", "10 / 0", "5 % 2", "2 + 3 + 4", "7 - 2"]:
    print(f"{expr!r:14} -> {calcular(expr)}")

'8 / 2'        -> 4.0
'3 * x'        -> Error de operandos: '3' y 'x' deben ser números válidos.
'10 / 0'       -> Error matemático: no se puede dividir entre cero.
'5 % 2'        -> Error de operador: '%' no está soportado. Usa +, -, * o /.
'2 + 3 + 4'    -> Error de formato: usa el formato '<numero> <operador> <numero>' (ejemplo: '8 / 2').
'7 - 2'        -> 5.0


In [56]:
class ProductoNoEncontradoError(Exception):
    """Se lanza cuando se consulta o retira un producto que no existe en el inventario."""
    def __init__(self, producto):
        self.producto = producto
        super().__init__(f"El producto '{producto}' no existe en el inventario.")


class Inventario:
    """Gestiona el stock de productos validando entradas y lanzando excepciones claras."""

    def __init__(self):
        self.productos = {}

    @staticmethod
    def _validar_nombre(producto):
        if not isinstance(producto, str):
            raise TypeError(f"El nombre del producto debe ser texto, no {type(producto).__name__}.")
        nombre = producto.strip()
        if not nombre:
            raise ValueError("El nombre del producto no puede estar vacío.")
        return nombre

    @staticmethod
    def _validar_cantidad(cantidad):
        if isinstance(cantidad, bool) or not isinstance(cantidad, int):
            raise TypeError(f"La cantidad debe ser un número entero, no {type(cantidad).__name__}.")
        if cantidad <= 0:
            raise ValueError("La cantidad debe ser mayor que cero.")
        return cantidad

    def añadir(self, producto, cantidad=1):
        nombre = self._validar_nombre(producto)
        cant = self._validar_cantidad(cantidad)
        self.productos[nombre] = self.productos.get(nombre, 0) + cant
        return self.productos[nombre]

    def retirar(self, producto, cantidad=1):
        nombre = self._validar_nombre(producto)
        cant = self._validar_cantidad(cantidad)
        if nombre not in self.productos:
            raise ProductoNoEncontradoError(nombre)
        if cant > self.productos[nombre]:
            raise ValueError(
                f"Stock insuficiente de '{nombre}': disponible {self.productos[nombre]}, solicitado {cant}."
            )
        self.productos[nombre] -= cant
        return self.productos[nombre]

    def consultar(self, producto):
        nombre = self._validar_nombre(producto)
        if nombre not in self.productos:
            raise ProductoNoEncontradoError(nombre)
        return self.productos[nombre]


# Pruebas de ejemplo
inv = Inventario()
print("Añadir pan (10):", inv.añadir("pan", 10))
print("Añadir pan (5):", inv.añadir("pan", 5))
print("Consultar pan:", inv.consultar("pan"))
print("Retirar pan (4):", inv.retirar("pan", 4))

for llamada in [
    lambda: inv.consultar("queso"),
    lambda: inv.retirar("pan", 99),
    lambda: inv.añadir("leche", -2),
    lambda: inv.añadir("", 3),
    lambda: inv.añadir("agua", True),
]:
    try:
        llamada()
    except (ProductoNoEncontradoError, ValueError, TypeError) as e:
        print(f"Capturado {type(e).__name__}: {e}")


Añadir pan (10): 10
Añadir pan (5): 15
Consultar pan: 15
Retirar pan (4): 11
Capturado ProductoNoEncontradoError: El producto 'queso' no existe en el inventario.
Capturado ValueError: Stock insuficiente de 'pan': disponible 11, solicitado 99.
Capturado ValueError: La cantidad debe ser mayor que cero.
Capturado ValueError: El nombre del producto no puede estar vacío.
Capturado TypeError: La cantidad debe ser un número entero, no bool.


## ✅ Autoevaluación

- [ ] **a)** Distingo un error de sintaxis (antes de ejecutar) de una excepción (en ejecución).
- [ ] **b)** Escribo `try`, `except`, `else` y `finally` y sé en qué orden se ejecutan.
- [ ] **c)** Capturo excepciones específicas y las trato (valor por defecto, reintento, aviso).
- [ ] **d)** Lanzo excepciones con `raise`, las relanzo y creo excepciones personalizadas.
- [ ] **e)** Sé leer un *traceback* y depurar con `print`, `logging`, `assert` y `pdb`.
- [ ] **f)** Escribo código robusto: valido entradas, libero recursos con `with`/`finally` y fallo con mensajes claros.

🎉 **Has completado el módulo de Estructuras de control.**